# Composite log-concave sampler: implementation and experiments

This notebook implements the Gibbs sampler in Algorithm 1 and the independent-Metropolis conditional sampler in Algorithm 2 of the draft.

The target family on $\mathbb{R}^d$ is

$$
\pi(x) \propto \exp\{-f(x)-g(x)\},
$$

where $f$ is smooth and strongly convex and $g$ is convex, possibly non-smooth.

We do the following experiments:

1. **Sparse Bayesian logistic regression with an $\ell_1$ prior.**  This is a non-quadratic target where the RGO for $g(x)=\lambda\|x\|_1$ is exact and coordinate-wise.
2. **Bayesian logistic regression over a box constraint.**  This is a non-Gaussian density restricted to a convex set. The RGO for the box is an exact product of truncated Gaussians.
3. **Gaussian restricted to a box**: this is a simple problem that investigates the scaling property of our algorithm.

The comparison methods are:

- **Composite sampler:** Algorithm 1 with Algorithm 2 used as the conditional sampler.
- **Prox-MALA:** a Metropolis-corrected proximal-gradient Gaussian proposal.
- **PGLA:** the unadjusted proximal-gradient Langevin algorithm, included because it is a natural fast baseline but has asymptotic discretization bias.

Step sizes are tuned by short pilot grids and the selected values are explicitly shown.

# Setup code environments

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
import math
import time
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from scipy.special import expit, log_ndtr, ndtr, ndtri, ndtri_exp

# Reproducibility and plot defaults.
SEED = 202602
rng_global = np.random.default_rng(SEED)

FIG_DIR = Path("composite_sampler_figures")
FIG_DIR.mkdir(exist_ok=True)
for old in FIG_DIR.glob("*"):
    old.unlink()

# Color palette
OKABE_ITO = {
    "black": "#000000",
    "orange": "#E69F00",
    "sky": "#56B4E9",
    "blue": "#0072B2",
    "vermillion": "#D55E00",
    "green": "#009E73",
    "yellow": "#F0E442",
    "purple": "#CC79A7",
    "gray": "#7F7F7F",
}

METHOD_STYLE = {
    "Reference": {"color": OKABE_ITO["black"], "marker": "o", "linestyle": "-", "hatch": ""},
    "Composite sampler": {"color": OKABE_ITO["blue"], "marker": "o", "linestyle": "-", "hatch": "///"},
    "Composite": {"color": OKABE_ITO["blue"], "marker": "o", "linestyle": "-", "hatch": "///"},
    "Prox-MALA": {"color": OKABE_ITO["orange"], "marker": "s", "linestyle": "--", "hatch": "\\\\"},
    "PGLA (no MH)": {"color": OKABE_ITO["vermillion"], "marker": "^", "linestyle": "-.", "hatch": "xx"},
    "PGLA": {"color": OKABE_ITO["vermillion"], "marker": "^", "linestyle": "-.", "hatch": "xx"},
    "Target atom": {"color": OKABE_ITO["gray"], "marker": "D", "linestyle": ":", "hatch": ".."},
}

plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "semibold",
    "legend.fontsize": 9,
    "figure.figsize": (6.6, 4.2),
    "figure.dpi": 150,
    "savefig.dpi": 350,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "grid.linewidth": 0.8,
    "grid.linestyle": "-",
    "lines.linewidth": 2.2,
    "lines.markersize": 5.5,
    "legend.frameon": True,
    "legend.framealpha": 0.95,
    "legend.edgecolor": "0.85",
    "legend.facecolor": "white",
    "xtick.direction": "out",
    "ytick.direction": "out",
})

PAPER_RUN = True

# Numerically stable truncated Gaussian sampler

The RGO for a box and the one-dimensional pieces of the $\ell^1$ RGO reduce to sampling

$$
Z \sim \mathcal N(\mu,\sigma^2)\,\big|\, [a,b].
$$

The helper below avoids subtracting nearly equal CDF values in the tails:

- left-tail intervals use log-CDF arithmetic;
- right-tail intervals use log-survival arithmetic, since $1-\Phi(t)=\Phi(-t)$;
- central intervals use ordinary inverse-CDF sampling.

This is the main numerical-stability point for the experiments.

In [ ]:
_EPS = np.finfo(float).tiny
_ONE_MINUS_EPS = 1.0 - np.finfo(float).eps


def _logdiffexp(a, b):
    """Return log(exp(a)-exp(b)) for a >= b, vectorized."""
    a, b = np.broadcast_arrays(a, b)
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        out = a + np.log1p(-np.exp(b - a))
    return np.where(np.isneginf(b), a, out)


def _logdiffexp_scalar(a: float, b: float) -> float:
    if np.isneginf(b):
        return float(a)
    if b >= a:
        return -np.inf
    return float(a + np.log1p(-np.exp(b - a)))


def truncnorm_scalar(mu: float, sigma: float, lower: float, upper: float, rng: np.random.Generator) -> float:
    """Sample N(mu, sigma^2) restricted to [lower, upper], scalar stable version."""
    if sigma <= 0:
        raise ValueError("sigma must be positive")
    if upper < lower:
        raise ValueError("upper must be >= lower")
    if upper == lower:
        return float(lower)

    lo = (lower - mu) / sigma
    hi = (upper - mu) / sigma
    u = float(np.clip(rng.random(), _EPS, _ONE_MINUS_EPS))

    if hi <= 0:  # left tail: use log Phi
        lp_lo = log_ndtr(lo)
        lp_hi = log_ndtr(hi)
        lmass = _logdiffexp_scalar(lp_hi, lp_lo)
        lp = np.logaddexp(lp_lo, np.log(u) + lmass)
        z = ndtri_exp(lp)
    elif lo >= 0:  # right tail: use log survival = log Phi(-x)
        ls_hi = log_ndtr(-hi)
        ls_lo = log_ndtr(-lo)
        lmass = _logdiffexp_scalar(ls_lo, ls_hi)
        ls = np.logaddexp(ls_hi, np.log(u) + lmass)
        z = -ndtri_exp(ls)
    else:  # central interval: direct inverse-CDF is stable
        p_lo = ndtr(lo)
        p_hi = ndtr(hi)
        p = np.clip(p_lo + u * (p_hi - p_lo), _EPS, _ONE_MINUS_EPS)
        z = ndtri(p)

    return float(mu + sigma * z)


def truncnorm_stable_rvs(mu, sigma, lower, upper, rng: np.random.Generator, size=None):
    """Vectorized stable sampler for N(mu, sigma^2) restricted to [lower, upper]."""
    mu = np.asarray(mu, dtype=float)
    sigma = np.asarray(sigma, dtype=float)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)

    if size is not None:
        mu = np.broadcast_to(mu, size)
        sigma = np.broadcast_to(sigma, size)
        lower = np.broadcast_to(lower, size)
        upper = np.broadcast_to(upper, size)

    mu, sigma, lower, upper = np.broadcast_arrays(mu, sigma, lower, upper)
    if np.any(sigma <= 0):
        raise ValueError("sigma must be positive")
    if np.any(upper < lower):
        raise ValueError("upper must be >= lower")

    shape = mu.shape
    u = np.clip(rng.random(shape), _EPS, _ONE_MINUS_EPS)
    lo = (lower - mu) / sigma
    hi = (upper - mu) / sigma
    z = np.empty(shape)

    point = upper == lower
    z[point] = lo[point]

    cdf_mask = (hi <= 0) & (~point)
    sf_mask = (lo >= 0) & (~point)
    central = (~cdf_mask) & (~sf_mask) & (~point)

    if np.any(cdf_mask):
        lp_lo = log_ndtr(lo[cdf_mask])
        lp_hi = log_ndtr(hi[cdf_mask])
        lmass = _logdiffexp(lp_hi, lp_lo)
        lp = np.logaddexp(lp_lo, np.log(u[cdf_mask]) + lmass)
        z[cdf_mask] = ndtri_exp(lp)

    if np.any(sf_mask):
        ls_hi = log_ndtr(-hi[sf_mask])
        ls_lo = log_ndtr(-lo[sf_mask])
        lmass = _logdiffexp(ls_lo, ls_hi)
        ls = np.logaddexp(ls_hi, np.log(u[sf_mask]) + lmass)
        z[sf_mask] = -ndtri_exp(ls)

    if np.any(central):
        p_lo = ndtr(lo[central])
        p_hi = ndtr(hi[central])
        p = np.clip(p_lo + u[central] * (p_hi - p_lo), _EPS, _ONE_MINUS_EPS)
        z[central] = ndtri(p)

    return mu + sigma * z


# Stress tests: extreme truncation intervals should remain finite and within support.
rng = np.random.default_rng(SEED)
for interval in [(8.0, 9.0), (-9.0, -8.0), (0.0, 1.0), (-np.inf, -7.5), (7.5, np.inf)]:
    draws = truncnorm_stable_rvs(0.0, 1.0, interval[0], interval[1], rng, size=(2_000,))
    assert np.all(np.isfinite(draws))
    assert np.all(draws >= interval[0]) and np.all(draws <= interval[1])
print("Stable truncated-normal checks passed.")

# RGO implementations

Two RGOs are used in the experiments.

For $g(x)=\lambda\|x\|_1$, separability reduces the sampler to one-dimensional densities. Completing the square gives two truncated-Gaussian pieces:

$$
p(x) \propto
\begin{cases}
\exp\left[-\dfrac{(x-(y+\lambda h))^2}{2h}\right], & x\le 0,\\[6pt]
\exp\left[-\dfrac{(x-(y-\lambda h))^2}{2h}\right], & x\ge 0.
\end{cases}
$$

For $g=\iota_{[a,b]^d}$, the RGO is a product of coordinate-wise truncated Gaussians.

In [ ]:
class L1RGO:
    """RGO for g(x)=lambda ||x||_1, sampled coordinate-wise exactly."""

    def __init__(self, lam: float):
        self.lam = float(lam)

    def sample_scalar(self, center: float, h: float, rng: np.random.Generator) -> float:
        lam = self.lam
        sh = math.sqrt(h)
        y = float(center)

        log_w_minus = lam * y + 0.5 * lam * lam * h + log_ndtr(-(y + lam * h) / sh)
        log_w_plus = -lam * y + 0.5 * lam * lam * h + log_ndtr((y - lam * h) / sh)
        p_minus = math.exp(log_w_minus - np.logaddexp(log_w_minus, log_w_plus))

        if rng.random() < p_minus:
            return truncnorm_scalar(y + lam * h, sh, -np.inf, 0.0, rng)
        return truncnorm_scalar(y - lam * h, sh, 0.0, np.inf, rng)

    def sample(self, center, h: float, rng: np.random.Generator):
        y = np.asarray(center, dtype=float)
        lam = self.lam
        sh = math.sqrt(h)

        log_w_minus = lam * y + 0.5 * lam * lam * h + log_ndtr(-(y + lam * h) / sh)
        log_w_plus = -lam * y + 0.5 * lam * lam * h + log_ndtr((y - lam * h) / sh)
        p_minus = np.exp(log_w_minus - np.logaddexp(log_w_minus, log_w_plus))
        choose_minus = rng.random(y.shape) < p_minus

        out = np.empty(y.shape)
        if np.any(choose_minus):
            out[choose_minus] = truncnorm_stable_rvs(
                y[choose_minus] + lam * h, sh, -np.inf, 0.0, rng
            )
        if np.any(~choose_minus):
            out[~choose_minus] = truncnorm_stable_rvs(
                y[~choose_minus] - lam * h, sh, 0.0, np.inf, rng
            )
        return out

    def g(self, x) -> float:
        return self.lam * float(np.sum(np.abs(x)))

    def prox(self, v, eta: float):
        return np.sign(v) * np.maximum(np.abs(v) - eta * self.lam, 0.0)


class BoxRGO:
    """RGO for the convex indicator of a box [lower, upper]."""

    def __init__(self, lower, upper):
        self.lower = np.asarray(lower, dtype=float)
        self.upper = np.asarray(upper, dtype=float)
        if self.lower.shape != self.upper.shape:
            raise ValueError("lower and upper must have the same shape")
        if np.any(self.upper < self.lower):
            raise ValueError("box upper bounds must exceed lower bounds")

    def sample(self, center, h: float, rng: np.random.Generator):
        return truncnorm_stable_rvs(np.asarray(center, dtype=float), math.sqrt(h), self.lower, self.upper, rng)

    def g(self, x) -> float:
        x = np.asarray(x)
        if np.all((x >= self.lower) & (x <= self.upper)):
            return 0.0
        return np.inf

    def prox(self, v, eta: float | None = None):
        return np.clip(v, self.lower, self.upper)

# Smooth potential, Data generation, and benchmarks

For both main experiments, the smooth part is a logistic negative log-likelihood plus a ridge term:

$$
f(x)
=
\sum_{i=1}^n
\left\{
\log\left(1+\exp(a_i^\top x)\right)
-y_i\,a_i^\top x
\right\}
+
\frac{\tau}{2}\|x\|_2^2 .
$$

It is $\tau$-strongly convex, and a valid smoothness bound is

$$
\beta \le \frac14 \|A\|_{\mathrm{op}}^2 + \tau .
$$

The composite sampler alternates

$$
Y_k\mid X_k=x \sim \mathcal N(x,hI)
$$

and an approximate draw from

$$
X_{k+1}\mid Y_k=y
\propto
\exp\left\{-f(x)-g(x)-\frac{1}{2h}\|x-y\|^2\right\}.
$$

Algorithm 2 implements this conditional with an independent-MH proposal based on the first-order model of $f$,

$$
\mu(\mathrm dx)
\propto
\exp\left\{
-g(x)-\frac{1}{2h}\left\|x-\left(y-h\nabla f(y)\right)\right\|^2
\right\}\,\mathrm dx,
$$

which only requires an RGO for $g$.

In [ ]:
@dataclass
class LogisticPotential:
    A: np.ndarray
    y: np.ndarray
    tau: float
    _op_norm_sq: float = field(init=False, repr=False)

    def __post_init__(self):
        self.A = np.asarray(self.A, dtype=float)
        self.y = np.asarray(self.y, dtype=float)
        self._op_norm_sq = float(np.linalg.norm(self.A, 2) ** 2)

    @property
    def dim(self) -> int:
        return self.A.shape[1]

    @property
    def alpha(self) -> float:
        return float(self.tau)

    @property
    def beta(self) -> float:
        return 0.25 * self._op_norm_sq + float(self.tau)

    def value(self, x) -> float:
        x = np.asarray(x, dtype=float)
        z = self.A @ x
        return float(np.sum(np.logaddexp(0.0, z) - self.y * z) + 0.5 * self.tau * np.dot(x, x))

    def grad(self, x):
        x = np.asarray(x, dtype=float)
        z = self.A @ x
        return self.A.T @ (expit(z) - self.y) + self.tau * x




In [ ]:
# log π
def log_target(f: LogisticPotential, rgo, x) -> float:
    gx = rgo.g(x)
    if not np.isfinite(gx):
        return -np.inf
    return -f.value(x) - gx

# Computes the mode of π
def solve_prox_map(f: LogisticPotential, rgo, beta: float | None = None, x0=None, n_iter: int = 5_000, tol: float = 1e-10):
    """FISTA for argmin_x f(x)+g(x), using only grad f and prox_g."""
    if beta is None:
        beta = f.beta
    x = np.zeros(f.dim) if x0 is None else np.asarray(x0, dtype=float).copy()
    z = x.copy()
    t = 1.0

    for _ in range(n_iter):
        x_old = x.copy()
        x = rgo.prox(z - f.grad(z) / beta, 1.0 / beta)
        t_new = 0.5 * (1.0 + math.sqrt(1.0 + 4.0 * t * t))
        z = x + ((t - 1.0) / t_new) * (x - x_old)
        t = t_new
        if np.linalg.norm(x - x_old) <= tol * (1.0 + np.linalg.norm(x_old)):
            break
    return x

# Algorithms for comparison (Composite Sampler, Prox-MALA, PGLA)

## Composite Sampler (this paper)

The following code block implements the algorithm in our paper, which combines Algorithm 1 and Algorithm 2.

In [ ]:
# Algorithm 2
def independent_mh_conditional(
    f: LogisticPotential,
    rgo,
    h: float,
    y: np.ndarray,
    inner_steps: int,
    rng: np.random.Generator,
):
    m = y - h * f.grad(y)

    # Algorithm 2 initializes from the proposal mu.
    x = rgo.sample(m, h, rng)
    if inner_steps <= 0:
        return x, np.nan

    fx = f.value(x)
    xy = float(np.sum((x - y) ** 2))
    xm = float(np.sum((x - m) ** 2))

    accepted = 0
    for _ in range(inner_steps):
        z = rgo.sample(m, h, rng)
        fz = f.value(z)
        zy = float(np.sum((z - y) ** 2))
        zm = float(np.sum((z - m) ** 2))

        log_alpha = (-fz - 0.5 * zy / h - 0.5 * xm / h) - (
            -fx - 0.5 * xy / h - 0.5 * zm / h
        )
        if math.log(max(rng.random(), _EPS)) <= min(0.0, log_alpha):
            x, fx, xy, xm = z, fz, zy, zm
            accepted += 1

    return x, accepted / inner_steps

# Algorithm 1, which calls Algorithm 2 at each iteration.
def composite_sampler(
    f: LogisticPotential,
    rgo,
    h: float,
    n_steps: int,
    inner_steps: int,
    x_star: np.ndarray,
    beta: float,
    rng: np.random.Generator,
    x0: np.ndarray | None = None,
    burn: int = 0,
    thin: int = 1,
):
    d = f.dim
    if x0 is None:
        x = rgo.sample(x_star, 1.0 / beta, rng)
    else:
        x = np.asarray(x0, dtype=float).copy()

    samples = []
    acceptances = []
    sqrt_h = math.sqrt(h)
    t0 = time.perf_counter()

    for k in range(n_steps):
        y = x + sqrt_h * rng.normal(size=d)
        x, acc = independent_mh_conditional(f, rgo, h, y, inner_steps, rng)
        acceptances.append(acc)
        if k >= burn and ((k - burn) % thin == 0):
            samples.append(x.copy())

    return {
        "samples": np.asarray(samples),
        "acceptance": np.asarray(acceptances),
        "elapsed": time.perf_counter() - t0,
        "h": h,
        "inner_steps": inner_steps,
        "work_per_sample": inner_steps + 1,
    }

## The proximal Metropolis-adjusted Langevin algorithm (Prox-MALA)

The iterates follow the proposal
$$
Q(x, \cdot) = \mathcal{N}(\text{prox}_{hg}(x - h \nabla f(x)), 2h I_d).
$$

In [ ]:
def prox_mala(
    f: LogisticPotential,
    rgo,
    step: float,
    n_steps: int,
    rng: np.random.Generator,
    x0: np.ndarray | None = None,
    burn: int = 0,
    thin: int = 1,
):
    """MH-corrected proximal-gradient Gaussian proposal."""
    d = f.dim
    x = np.zeros(d) if x0 is None else np.asarray(x0, dtype=float).copy()
    if not np.isfinite(log_target(f, rgo, x)):
        x = rgo.prox(x, step)

    logp = log_target(f, rgo, x)
    samples = []
    acceptances = []
    proposal_feasible = []
    sqrt_2step = math.sqrt(2.0 * step)
    t0 = time.perf_counter()

    for k in range(n_steps):
        mean_x = rgo.prox(x - step * f.grad(x), step)
        z = mean_x + sqrt_2step * rng.normal(size=d)
        logp_z = log_target(f, rgo, z)
        proposal_feasible.append(float(np.isfinite(logp_z)))

        if np.isfinite(logp_z):
            mean_z = rgo.prox(z - step * f.grad(z), step)
            log_alpha = (
                logp_z
                - np.sum((x - mean_z) ** 2) / (4.0 * step)
                - logp
                + np.sum((z - mean_x) ** 2) / (4.0 * step)
            )
        else:
            log_alpha = -np.inf

        if math.log(max(rng.random(), _EPS)) <= min(0.0, log_alpha):
            x, logp = z, logp_z
            acceptances.append(1.0)
        else:
            acceptances.append(0.0)

        if k >= burn and ((k - burn) % thin == 0):
            samples.append(x.copy())

    return {
        "samples": np.asarray(samples),
        "acceptance": np.asarray(acceptances),
        "proposal_feasible": np.asarray(proposal_feasible),
        "elapsed": time.perf_counter() - t0,
        "step": step,
        "work_per_sample": 2,  # forward and reverse gradients
    }

## The proximal gradient Langevin algorithm (PGLA)

The iterates are
$$
x_{k+1} = \text{prox}_{hg} (x_k - h \nabla f(x_k) + \mathcal{N}(0, 2 h I_d))
$$

In [ ]:
def proximal_gradient_langevin(
    f: LogisticPotential,
    rgo,
    step: float,
    n_steps: int,
    rng: np.random.Generator,
    x0: np.ndarray | None = None,
    burn: int = 0,
    thin: int = 1,
):
    """Unadjusted proximal-gradient Langevin algorithm."""
    d = f.dim
    x = np.zeros(d) if x0 is None else np.asarray(x0, dtype=float).copy()
    if not np.isfinite(rgo.g(x)):
        x = rgo.prox(x, step)

    samples = []
    sqrt_2step = math.sqrt(2.0 * step)
    t0 = time.perf_counter()

    for k in range(n_steps):
        x = rgo.prox(x - step * f.grad(x) + sqrt_2step * rng.normal(size=d), step)
        if k >= burn and ((k - burn) % thin == 0):
            samples.append(x.copy())

    return {
        "samples": np.asarray(samples),
        "elapsed": time.perf_counter() - t0,
        "step": step,
        "work_per_sample": 1,
    }

# Metrics, tuning, and plotting helpers

## Experiment 1 & 2

The primary scalar diagnostic for Experiment 1 & 2 is root mean-squared error of the running posterior mean relative to a long, conservative Prox-MALA reference:
$$
\operatorname{RMSE}(t)
=
d^{-1/2}\|\bar x_t-\widehat m_{\mathrm{ref}}\|_2 .
$$
This diagnostic is useful but incomplete: an unadjusted chain such as PGLA can have low posterior-mean RMSE while still targeting the wrong stationary distribution. The notebook therefore also tracks target-specific artifacts:

- for $\ell^1$ targets, the true density is continuous and has no atom at zero, while the PGLA soft-thresholding step can create exact zeros;
- for box-constrained targets, the true truncated density has no boundary atoms, while projected PGLA can create exact boundary hits;
- for Prox-MALA on a constrained set, the Gaussian proposal can waste many proposals outside the box, which is recorded as the feasible-proposal rate.

Step sizes are selected by short pilot grids. The tuning figures show every tested multiplier and mark the selected value with a star.

## Experiment 3

In Experiement 3, we use (sliced) Wasserstein-2 metric, which is a way to measure how different two multidimensional sample distributions are. The sliced version makes high-dimensional problem easier by picking many random unit directions, projecting both samples onto these directions, and then computing and averaging the 1D Wasserstein-2 distance between the projected samples.

Unlike the running MSE for posterior mean, it is more sensitive to mean, variance, and tail shape, so it actually penalizes PGLA's asymptotic bias.

## Metrics

In [ ]:
def running_mean_rmse(samples: np.ndarray, ref_mean: np.ndarray):
    samples = np.asarray(samples, dtype=float)
    ref_mean = np.asarray(ref_mean, dtype=float)
    denom = np.arange(1, len(samples) + 1, dtype=float).reshape(-1, 1)
    cmean = np.cumsum(samples, axis=0) / denom
    return np.sqrt(np.mean((cmean - ref_mean) ** 2, axis=1))


def mean_rmse(samples: np.ndarray, ref_mean: np.ndarray) -> float:
    samples = np.asarray(samples, dtype=float)
    ref_mean = np.asarray(ref_mean, dtype=float)
    return float(np.sqrt(np.mean((samples.mean(axis=0) - ref_mean) ** 2)))


def variance_rmse(samples: np.ndarray, ref_samples: np.ndarray) -> float:
    samples = np.asarray(samples, dtype=float)
    ref_samples = np.asarray(ref_samples, dtype=float)
    return float(np.sqrt(np.mean((samples.var(axis=0) - ref_samples.var(axis=0)) ** 2)))

def sliced_w2(samples_a, samples_b, projections, n_quantiles=200):
    """Sliced W2 distance, given pre-drawn unit projections of shape (d, n_proj)."""
    proj_a = samples_a @ projections
    proj_b = samples_b @ projections
    q = (np.arange(n_quantiles) + 0.5) / n_quantiles
    qa = np.quantile(proj_a, q, axis=0)
    qb = np.quantile(proj_b, q, axis=0)
    return float(np.sqrt(np.mean((qa - qb) ** 2)))


def running_sliced_w2(samples, ref_samples, n_proj=64, n_quantiles=200,
                      eval_grid=None, rng=None):
    """Sliced W2 of samples[:k] against ref_samples for k on a log-grid.

    The same projections are used at every k so the curve has a consistent
    metric. Returns (eval_grid, w2_values).
    """
    rng = rng or np.random.default_rng(0)
    n, d = samples.shape
    P = rng.standard_normal((d, n_proj))
    P /= np.linalg.norm(P, axis=0, keepdims=True)
    if eval_grid is None:
        eval_grid = np.unique(np.geomspace(max(50, n // 200), n, 24).astype(int))
    eval_grid = np.asarray(eval_grid)
    eval_grid = eval_grid[(eval_grid <= n) & (eval_grid >= 20)]
    out = np.empty(len(eval_grid))
    for i, k in enumerate(eval_grid):
        out[i] = sliced_w2(samples[:k], ref_samples, P, n_quantiles=n_quantiles)
    return eval_grid, out

In [ ]:
def acceptance_mean(result: dict) -> float:
    if "acceptance" not in result:
        return np.nan
    vals = np.asarray(result["acceptance"], dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return np.nan
    return float(vals.mean())


def feasible_mean(result: dict) -> float:
    if "proposal_feasible" not in result:
        return np.nan
    vals = np.asarray(result["proposal_feasible"], dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return np.nan
    return float(vals.mean())


def as_1d_numpy(x, dtype=None):
    """Return a 1D NumPy array, avoiding pandas Series multi-indexing pitfalls in Matplotlib."""
    return np.asarray(x, dtype=dtype).reshape(-1)

def autocorr_fft(x: np.ndarray):
    x = np.asarray(x, dtype=float)
    n = len(x)
    x = x - x.mean()
    if n < 2 or np.allclose(x.var(), 0):
        return np.ones(1)
    size = 1 << (2 * n - 1).bit_length()
    fx = np.fft.rfft(x, n=size)
    acf = np.fft.irfft(fx * np.conjugate(fx), n=size)[:n]
    acf /= acf[0]
    return acf


def ess_1d(x: np.ndarray, max_lag: int | None = None):
    acf = autocorr_fft(x)
    n = len(acf)
    if max_lag is None:
        max_lag = min(n - 1, 500)
    tau = 1.0
    for lag in range(1, max_lag + 1, 2):
        pair = acf[lag]
        if lag + 1 <= max_lag:
            pair += acf[lag + 1]
        if pair <= 0:
            break
        tau += 2.0 * pair
    return max(1.0, len(x) / tau)


def median_ess(samples: np.ndarray):
    samples = np.asarray(samples, dtype=float)
    return float(np.median([ess_1d(samples[:, j]) for j in range(samples.shape[1])]))


def exact_zero_fraction(samples: np.ndarray, tol: float = 1e-12) -> float:
    samples = np.asarray(samples, dtype=float)
    return float(np.mean(np.abs(samples) <= tol))


def boundary_hit_fraction(samples: np.ndarray, box: BoxRGO, tol: float = 1e-12) -> float:
    samples = np.asarray(samples, dtype=float)
    lower = np.asarray(box.lower, dtype=float).reshape(1, -1)
    upper = np.asarray(box.upper, dtype=float).reshape(1, -1)
    lower_hits = np.isclose(samples, lower, atol=tol, rtol=0.0)
    upper_hits = np.isclose(samples, upper, atol=tol, rtol=0.0)
    return float(np.mean(lower_hits | upper_hits))


def summarize_chain(name: str, result: dict, ref_samples: np.ndarray, ref_mean: np.ndarray, artifact: str | None = None, box: BoxRGO | None = None):
    samples = result["samples"]
    row = {
        "method": name,
        "samples": len(samples),
        "mean RMSE": mean_rmse(samples, ref_mean),
        "variance RMSE": variance_rmse(samples, ref_samples),
        "median ESS": median_ess(samples) if len(samples) > 20 else np.nan,
        "acceptance": acceptance_mean(result),
        "feasible proposal rate": feasible_mean(result),
        "elapsed sec": result.get("elapsed", np.nan),
    }
    if "h" in result:
        row["step or h"] = result["h"]
    elif "step" in result:
        row["step or h"] = result["step"]
    else:
        row["step or h"] = np.nan

    if artifact == "l1_zero":
        row["exact-zero fraction"] = exact_zero_fraction(samples)
    if artifact == "box_boundary":
        row["boundary-hit fraction"] = boundary_hit_fraction(samples, box)
    return row


def run_tuning_grid(method: str, multipliers, runner, ref_mean: np.ndarray, min_acceptance: float | None = None):
    rows = []
    for i, mult in enumerate(multipliers):
        result = runner(float(mult), i)
        acc = acceptance_mean(result)
        rmse = mean_rmse(result["samples"], ref_mean)
        eligible = True
        if min_acceptance is not None and np.isfinite(acc):
            eligible = acc >= min_acceptance
        rows.append({
            "method": method,
            "multiplier": float(mult),
            "pilot mean RMSE": rmse,
            "pilot acceptance": acc,
            "pilot feasible rate": feasible_mean(result),
            "eligible": eligible,
            "elapsed sec": result["elapsed"],
        })
    df = pd.DataFrame(rows)
    eligible_df = df[df["eligible"]]
    if len(eligible_df) == 0:
        eligible_df = df
    selected_idx = eligible_df["pilot mean RMSE"].idxmin()
    df["selected"] = False
    df.loc[selected_idx, "selected"] = True
    selected_multiplier = float(df.loc[selected_idx, "multiplier"])
    return df, selected_multiplier


## Plotting helpers

In [ ]:
def style_for(name: str, fallback_index: int = 0) -> dict:
    if name in METHOD_STYLE:
        return METHOD_STYLE[name]
    palette = [OKABE_ITO["blue"], OKABE_ITO["orange"], OKABE_ITO["green"], OKABE_ITO["purple"], OKABE_ITO["vermillion"]]
    markers = ["o", "s", "^", "D", "v"]
    linestyles = ["-", "--", "-.", ":", (0, (5, 2, 1, 2))]
    return {
        "color": palette[fallback_index % len(palette)],
        "marker": markers[fallback_index % len(markers)],
        "linestyle": linestyles[fallback_index % len(linestyles)],
        "hatch": "",
    }


def prettify_axes(ax, *, title: str | None = None, xlog: bool = False, ylog: bool = False):
    if title is not None:
        ax.set_title(title, pad=9)
    if xlog:
        ax.set_xscale("log")
        ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%g"))
    if ylog:
        ax.set_yscale("log")
    ax.grid(True, alpha=0.22, which="major")
    if xlog or ylog:
        ax.grid(True, alpha=0.12, which="minor")
    ax.tick_params(length=4, width=0.9)
    return ax


def save_figure(fig, filename: str):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{filename}.pdf")
    fig.savefig(FIG_DIR / f"{filename}.png")
    plt.show()

def plot_tuning(tuning_df: pd.DataFrame, title: str, filename: str):
    fig, ax = plt.subplots(figsize=(6.8, 4.3))
    for idx, (method, sub) in enumerate(tuning_df.groupby("method", sort=False)):
        sub = sub.sort_values("multiplier")
        st = style_for(method, idx)
        ax.plot(
            as_1d_numpy(sub["multiplier"], dtype=float),
            as_1d_numpy(sub["pilot mean RMSE"], dtype=float),
            marker=st["marker"],
            color=st["color"],
            linestyle=st["linestyle"],
            lw=2.3,
            label=method,
        )
        chosen = sub[sub["selected"]]
        if len(chosen):
            ax.scatter(
                as_1d_numpy(chosen["multiplier"], dtype=float),
                as_1d_numpy(chosen["pilot mean RMSE"], dtype=float),
                marker="*",
                s=190,
                color=st["color"],
                edgecolor="black",
                linewidth=0.8,
                zorder=5,
            )
    ax.set_xlabel(r"step multiplier in $h$ or $\eta$ = multiplier / $\beta$")
    ax.set_ylabel("pilot mean RMSE")
    prettify_axes(ax, title=title, xlog=True, ylog=True)
    ax.legend(title="method", loc="best")
    save_figure(fig, filename)


def plot_running_rmse(results, ref_mean, title, filename):
    fig, ax = plt.subplots(figsize=(6.8, 4.3))
    for idx, (name, result) in enumerate(results):
        samples = result["samples"]
        y = running_mean_rmse(samples, ref_mean)
        x = np.arange(1, len(y) + 1) * result.get("work_per_sample", 1)
        st = style_for(name, idx)
        ax.plot(x, y, lw=2.4, label=name, color=st["color"], linestyle=st["linestyle"])
        # Sparse markers make the curves distinguishable in grayscale too.
        if len(x) > 5:
            mark_idx = np.unique(np.linspace(0, len(x) - 1, min(8, len(x)), dtype=int))
            ax.plot(x[mark_idx], y[mark_idx], linestyle="None", marker=st["marker"], color=st["color"], markersize=5.2)
    ax.set_xlabel("gradient evaluations")
    ax.set_ylabel("running mean RMSE")
    prettify_axes(ax, title=title, xlog=True, ylog=True)
    ax.legend(title="method", loc="best")
    save_figure(fig, filename)

def plot_running_metric(results, ref_samples, metric_fn, ylabel, title,
                        filename, **metric_kwargs):
    """Generic running-metric plotter parameterized by metric_fn(samples, ref)."""
    fig, ax = plt.subplots(figsize=(6.8, 4.3))
    for idx, (name, result) in enumerate(results):
        grid, vals = metric_fn(result["samples"], ref_samples, **metric_kwargs)
        x = grid * result.get("work_per_sample", 1)
        st = style_for(name, idx)
        ax.plot(x, vals, lw=2.4, label=name,
                color=st["color"], linestyle=st["linestyle"],
                marker=st["marker"], markersize=5.5,
                markevery=max(1, len(x) // 8))
    ax.set_xlabel("gradient evaluations")
    ax.set_ylabel(ylabel)
    prettify_axes(ax, title=title, xlog=True, ylog=True)
    ax.legend(title="method", loc="best")
    save_figure(fig, filename)

def plot_metric_bars(summary_df: pd.DataFrame, metric: str, title: str, filename: str, ylabel: str | None = None, log: bool = False, ylim=None):
    df = summary_df.copy()
    values = as_1d_numpy(df[metric], dtype=float)
    labels = list(df["method"])
    x = np.arange(len(labels))
    fig, ax = plt.subplots(figsize=(6.6, 4.2))
    for i, (label, value) in enumerate(zip(labels, values)):
        st = style_for(label, i)
        ax.bar(
            x[i],
            value,
            width=0.72,
            color=st["color"],
            edgecolor="black",
            linewidth=0.8,
            hatch=st.get("hatch", ""),
            alpha=0.92,
        )
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=12, ha="right")
    ax.set_ylabel(ylabel or metric)
    if ylim is not None:
        ax.set_ylim(*ylim)
    prettify_axes(ax, title=title, ylog=log)
    for i, value in enumerate(values):
        if np.isfinite(value):
            label = f"{value:.2g}" if (abs(value) < 0.01 or abs(value) >= 100) else f"{value:.3f}"
            ax.annotate(label, (x[i], value), xytext=(0, 4), textcoords="offset points", ha="center", va="bottom", fontsize=8.5)
    save_figure(fig, filename)


def plot_bias_tradeoff(df: pd.DataFrame, artifact_col: str, title: str, filename: str, artifact_label: str):
    fig, ax1 = plt.subplots(figsize=(6.8, 4.3))
    x = as_1d_numpy(df["multiplier"], dtype=float)
    y = as_1d_numpy(df["mean RMSE"], dtype=float)
    art = as_1d_numpy(df[artifact_col], dtype=float)

    ax1.plot(x, y, color=OKABE_ITO["blue"], marker="o", linestyle="-", label="mean RMSE")
    ax1.set_xlabel(r"PGLA step multiplier $\eta\beta$")
    ax1.set_ylabel("mean RMSE", color=OKABE_ITO["blue"])
    ax1.tick_params(axis="y", labelcolor=OKABE_ITO["blue"])
    prettify_axes(ax1, title=title, xlog=True, ylog=True)

    ax2 = ax1.twinx()
    ax2.plot(x, art, color=OKABE_ITO["vermillion"], marker="^", linestyle="--", label=artifact_label)
    ax2.set_ylabel(artifact_label, color=OKABE_ITO["vermillion"])
    ax2.tick_params(axis="y", labelcolor=OKABE_ITO["vermillion"])
    ax2.grid(False)
    ax2.set_ylim(bottom=0.0)

    if "selected" in df.columns and df["selected"].any():
        selected = df[df["selected"]]
        sx = float(selected["multiplier"].iloc[0])
        ax1.axvline(sx, color="0.2", linestyle=":", linewidth=1.5, label="_selected_step")
        ax1.text(sx, ax1.get_ylim()[0], " selected", rotation=90, va="bottom", ha="left", fontsize=8.5)

    handles = []
    labels = []
    for h in ax1.get_lines() + ax2.get_lines():
        label = h.get_label()
        if not label.startswith("_"):
            handles.append(h)
            labels.append(label)
    ax1.legend(handles, labels, loc="best")
    save_figure(fig, filename)


def save_current_figure(filename: str):
    fig = plt.gcf()
    save_figure(fig, filename)


def build_reference_prox_mala(f, rgo, x_star, n_steps: int, step_multiplier: float, seeds):
    """Pool long conservative Prox-MALA chains to estimate reference moments."""
    chains = []
    elapsed = 0.0
    acceptances = []
    feasible = []
    burn = max(800, n_steps // 5)
    for seed in seeds:
        out = prox_mala(
            f,
            rgo,
            step=step_multiplier / f.beta,
            n_steps=n_steps,
            rng=np.random.default_rng(seed),
            x0=x_star,
            burn=burn,
            thin=2,
        )
        chains.append(out["samples"])
        elapsed += out["elapsed"]
        acceptances.append(out["acceptance"])
        if "proposal_feasible" in out:
            feasible.append(out["proposal_feasible"])
    ret = {
        "samples": np.vstack(chains),
        "acceptance": np.concatenate(acceptances),
        "elapsed": elapsed,
        "step": step_multiplier / f.beta,
        "work_per_sample": 2,
    }
    if feasible:
        ret["proposal_feasible"] = np.concatenate(feasible)
    return ret

def plot_running_w2_at_d(curves_df, target_d, threshold, title, filename):
    """Sliced W2 vs gradient evaluations at one d, all methods, median over seeds."""
    sub = curves_df[curves_df["d"] == target_d]
    if len(sub) == 0:
        available = sorted(curves_df["d"].unique())
        raise ValueError(f"target_d={target_d} not in curves_df; available d's: {available}")
    fig, ax = plt.subplots(figsize=(6.8, 4.3))
    for idx, name in enumerate(["Composite sampler", "Prox-MALA", "PGLA (no MH)"]):
        ms = sub[sub["method"] == name]
        agg = (ms.groupby("grad_evals")["sliced_w2"].median()
               .reset_index().sort_values("grad_evals"))
        st = style_for(name, idx)
        ax.plot(agg["grad_evals"], agg["sliced_w2"], lw=2.4, label=name,
                color=st["color"], linestyle=st["linestyle"],
                marker=st["marker"], markersize=5.5,
                markevery=max(1, len(agg) // 8))
    ax.axhline(threshold, color="0.45", linestyle=":", lw=1.3,
               label=f"threshold = {threshold}")
    ax.set_xlabel("gradient evaluations")
    ax.set_ylabel("sliced Wasserstein-2 distance")
    prettify_axes(ax, title=title, xlog=True, ylog=True)
    ax.legend(loc="best", fontsize=9)
    save_figure(fig, filename)


def plot_cost_vs_d_slope(rows_df, threshold, title, filename, fit_min_d=None):
    """Cost-to-threshold against d on EXPLICIT log-log axes.
       x = log d, y = log(grad evaluations).

    Methods that fail to reach threshold show as hollow markers near the top.
    """
    fig, ax = plt.subplots(figsize=(7.0, 4.6))
    summary = []

    all_finite = rows_df.dropna(subset=["cost_to_threshold"])
    miss_y = (
        np.log(all_finite["cost_to_threshold"].max()) + 1.5
        if len(all_finite)
        else 14.0
    )

    for idx, name in enumerate(["Composite sampler", "Prox-MALA", "PGLA"]):
        sub = rows_df[rows_df["method"] == name].copy()
        agg = sub.groupby("d")["cost_to_threshold"].median().reset_index()
        finite = agg.dropna()
        st = style_for(name, idx)

        if len(finite):
            log_d = np.log(finite["d"].values.astype(float))
            log_c = np.log(finite["cost_to_threshold"].values.astype(float))

            ax.plot(
                log_d,
                log_c,
                lw=2.2,
                color=st["color"],
                linestyle=st["linestyle"],
                marker=st["marker"],
                markersize=8,
                label=name,
            )

            if fit_min_d is not None:
                mask = finite["d"].values >= fit_min_d
                fit_d, fit_c = log_d[mask], log_c[mask]
            else:
                fit_d, fit_c = log_d, log_c

            # Compute slope for returned summary only.
            # No fitted line or slope annotation is plotted.
            if len(fit_d) >= 2:
                slope, intercept = np.polyfit(fit_d, fit_c, 1)
                summary.append({
                    "method": name,
                    "fitted_slope": slope,
                    "intercept": intercept,
                })
            else:
                summary.append({
                    "method": name,
                    "fitted_slope": np.nan,
                    "intercept": np.nan,
                })

        missed = agg[agg["cost_to_threshold"].isna()]
        if len(missed):
            log_d_missed = np.log(missed["d"].values.astype(float))
            ax.scatter(
                log_d_missed,
                np.full(len(missed), miss_y),
                facecolors="none",
                edgecolors=st["color"],
                marker=st["marker"],
                s=85,
                linewidth=1.6,
                label=name if not len(finite) else None,
            )

    prettify_axes(ax, title=title, xlog=False, ylog=False)

    ax.set_ylabel(
        rf"$\log$(gradient evaluations to reach W2 $\leq$ {threshold})"
    )

    d_ticks = sorted({int(d) for d in rows_df["d"].unique()})
    ax.set_xticks(np.log(d_ticks))
    ax.set_xticklabels(
        [f"{d}\n({np.log(d):.2f})" for d in d_ticks],
        fontsize=8.5,
    )
    ax.set_xlabel(r"$d$  (with $\log d$ below)")

    ax.legend(loc="best", fontsize=9)

    save_figure(fig, filename)

    return pd.DataFrame(summary)

# Experiment 1: Sparse Bayesian logistic regression with an $\ell_1$ prior

The target is

$$
\pi(\theta)
\propto
\exp\left[
-\sum_{i=1}^n
\left\{
\log(1+\exp(a_i^\top\theta))
-y_i a_i^\top\theta
\right\}
-\frac{\tau}{2}\|\theta\|_2^2
-\lambda\|\theta\|_1
\right].
$$

This is a natural composite target: the likelihood is non-quadratic and smooth, while the Laplace prior contributes a non-smooth separable term.  I use a moderately strong $\ell_1$ penalty so that the nonsmooth part is not a cosmetic addition.  This is also where PGLA's soft-thresholding bias is easiest to see: the true posterior has no point mass at $\theta_j=0$, but unadjusted PGLA can produce exact zeros.

In [ ]:
def make_sparse_logistic_l1_problem(seed=10, paper_run=False):
    rng = np.random.default_rng(seed)
    if paper_run:
        d, n, n_ref, n_main, n_pilot = 36, 360, 16_000, 6_000, 1_200
    else:
        d, n, n_ref, n_main, n_pilot = 24, 220, 3_500, 1_600, 350

    rho = 0.65
    idx = np.arange(d)
    cov = rho ** np.abs(idx.reshape(-1, 1) - idx.reshape(1, -1))
    A = rng.multivariate_normal(np.zeros(d), cov, size=n)
    A = A / np.linalg.norm(A, axis=0) * math.sqrt(n)

    theta_true = np.zeros(d)
    support = np.arange(min(5, d))
    signs = np.array([1.0, -1.0, 0.8, 1.2, -0.9])[: len(support)]
    theta_true[support] = signs

    y = rng.binomial(1, expit(A @ theta_true))

    # Strong enough to make the nonsmooth prior visible, but still a standard sparse logistic model.
    tau = 0.20
    #tau = 0
    lam = 7.0

    f = LogisticPotential(A, y, tau)
    rgo = L1RGO(lam)
    x_star = solve_prox_map(f, rgo, f.beta, n_iter=4_000)
    return f, rgo, x_star, theta_true, lam, n_ref, n_main, n_pilot


f_l1, rgo_l1, x_star_l1, theta_true_l1, lam_l1, n_ref_l1, n_main_l1, n_pilot_l1 = make_sparse_logistic_l1_problem(
    seed=43, paper_run=PAPER_RUN
)
print(f"d={f_l1.dim}, alpha={f_l1.alpha:.3g}, beta={f_l1.beta:.3g}, kappa={f_l1.beta/f_l1.alpha:.1f}, lambda={lam_l1}")
print(f"MAP nonzeros: {np.sum(np.abs(x_star_l1) > 1e-10)} / {f_l1.dim}")

ref_l1 = build_reference_prox_mala(
    f_l1, rgo_l1, x_star_l1, n_steps=n_ref_l1, step_multiplier=0.20, seeds=[101, 102, 103]
)
ref_l1_samples = ref_l1["samples"]
ref_l1_mean = ref_l1_samples.mean(axis=0)
print(
    f"Reference: {len(ref_l1_samples)} retained samples, "
    f"acceptance={acceptance_mean(ref_l1):.3f}, elapsed={ref_l1['elapsed']:.2f}s"
)

In [ ]:
inner_steps_l1 = 8
composite_grid = [0.25, 0.5, 1.0, 2.0, 4.0, 8.0]
langevin_grid = [0.025, 0.05, 0.10, 0.20, 0.40, 0.80, 1.60]

tune_comp_l1, mult_comp_l1 = run_tuning_grid(
    "Composite sampler",
    composite_grid,
    lambda mult, i: composite_sampler(
        f_l1,
        rgo_l1,
        h=mult / f_l1.beta,
        n_steps=n_pilot_l1,
        inner_steps=inner_steps_l1,
        x_star=x_star_l1,
        beta=f_l1.beta,
        rng=np.random.default_rng(200 + i),
    ),
    ref_l1_mean,
    min_acceptance=0.15,
)

tune_pmala_l1, mult_pmala_l1 = run_tuning_grid(
    "Prox-MALA",
    langevin_grid,
    lambda mult, i: prox_mala(
        f_l1,
        rgo_l1,
        step=mult / f_l1.beta,
        n_steps=n_pilot_l1,
        rng=np.random.default_rng(300 + i),
        x0=x_star_l1,
    ),
    ref_l1_mean,
    min_acceptance=0.15,
)

tune_pgla_l1, mult_pgla_l1 = run_tuning_grid(
    "PGLA (no MH)",
    langevin_grid,
    lambda mult, i: proximal_gradient_langevin(
        f_l1,
        rgo_l1,
        step=mult / f_l1.beta,
        n_steps=n_pilot_l1,
        rng=np.random.default_rng(400 + i),
        x0=x_star_l1,
    ),
    ref_l1_mean,
)

tuning_l1 = pd.concat([tune_comp_l1, tune_pmala_l1, tune_pgla_l1], ignore_index=True)
plot_tuning(tuning_l1, "Step-size tuning: sparse logistic $\ell^1$ target", "logistic_l1_step_tuning")
tuning_l1

In [ ]:
comp_l1 = composite_sampler(
    f_l1,
    rgo_l1,
    h=mult_comp_l1 / f_l1.beta,
    n_steps=n_main_l1,
    inner_steps=inner_steps_l1,
    x_star=x_star_l1,
    beta=f_l1.beta,
    rng=np.random.default_rng(500),
)

pmala_l1 = prox_mala(
    f_l1,
    rgo_l1,
    step=mult_pmala_l1 / f_l1.beta,
    n_steps=n_main_l1,
    rng=np.random.default_rng(501),
    x0=x_star_l1,
)

pgla_l1 = proximal_gradient_langevin(
    f_l1,
    rgo_l1,
    step=mult_pgla_l1 / f_l1.beta,
    n_steps=n_main_l1,
    rng=np.random.default_rng(502),
    x0=x_star_l1,
)

summary_l1 = pd.DataFrame(
    [
        summarize_chain("Composite sampler", comp_l1, ref_l1_samples, ref_l1_mean, artifact="l1_zero"),
        summarize_chain("Prox-MALA", pmala_l1, ref_l1_samples, ref_l1_mean, artifact="l1_zero"),
        summarize_chain("PGLA (no MH)", pgla_l1, ref_l1_samples, ref_l1_mean, artifact="l1_zero"),
    ]
)
summary_l1.to_json("Sparse_logistic_l1__summary.json", orient="records", indent=2)
summary_l1

In [ ]:
plot_running_rmse(
    [
        ("Composite sampler", comp_l1),
        ("Prox-MALA", pmala_l1),
        ("PGLA (no MH)", pgla_l1),
    ],
    ref_l1_mean,
    title="Sparse logistic $\ell_1$: posterior mean error",
    filename="logistic_l1_mean_rmse",
)

plot_metric_bars(
    summary_l1,
    "exact-zero fraction",
    title="Sparse logistic $\ell_1$: exactly zero coordinates",
    filename="logistic_l1_zero_artifact",
    ylabel="fraction of coordinates exactly zero",
    ylim=(0.0, max(0.02, 1.15 * float(summary_l1["exact-zero fraction"].max()))),
)

ref_q05, ref_q95 = np.quantile(ref_l1_samples, [0.05, 0.95], axis=0)
comp_q05, comp_q95 = np.quantile(comp_l1["samples"], [0.05, 0.95], axis=0)
comp_mean = comp_l1["samples"].mean(axis=0)

j = np.arange(f_l1.dim)
fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.vlines(j - 0.10, ref_q05, ref_q95, color=OKABE_ITO["black"], lw=2.1, label="reference 90% interval")
ax.vlines(j + 0.10, comp_q05, comp_q95, color=OKABE_ITO["blue"], lw=2.1, label="composite 90% interval")
ax.scatter(j - 0.10, ref_l1_mean, s=18, color=OKABE_ITO["black"], marker="o")
ax.scatter(j + 0.10, comp_mean, s=24, color=OKABE_ITO["blue"], marker="D")
ax.axhline(0.0, color="0.45", lw=1.0, ls=":")
ax.set_xlabel("coefficient index")
ax.set_ylabel("posterior coefficient value")
prettify_axes(ax, title="Sparse logistic $\ell_1$: marginal posterior intervals")
ax.legend(loc="best")
save_figure(fig, "logistic_l1_intervals")

bias_rows = []
n_bias_l1 = n_pilot_l1 if not PAPER_RUN else 2 * n_pilot_l1
for i, mult in enumerate(langevin_grid):
    res = proximal_gradient_langevin(
        f_l1,
        rgo_l1,
        step=mult / f_l1.beta,
        n_steps=n_bias_l1,
        rng=np.random.default_rng(1500 + i),
        x0=x_star_l1,
    )
    bias_rows.append({
        "multiplier": float(mult),
        "mean RMSE": mean_rmse(res["samples"], ref_l1_mean),
        "exact-zero fraction": exact_zero_fraction(res["samples"]),
        "selected": float(mult) == float(mult_pgla_l1),
    })
pgla_bias_l1 = pd.DataFrame(bias_rows)
pgla_bias_l1.to_json("Sparse_logistic_l1__pgla_bias_curve.json", orient="records", indent=2)
plot_bias_tradeoff(
    pgla_bias_l1,
    "exact-zero fraction",
    title="Sparse logistic $\ell_1$: PGLA bias grows with step size",
    filename="logistic_l1_pgla_bias_tradeoff",
    artifact_label="exact-zero fraction",
)
pgla_bias_l1

# Experiment 2: Bayesian logistic regression over a box constraint

The box example uses a non-quadratic density,

$$
\pi(\theta)
\propto
\exp\left[
-\sum_{i=1}^n
\left\{
\log(1+\exp(a_i^\top\theta))
-y_i a_i^\top\theta
\right\}
-\frac{\tau}{2}\|\theta\|_2^2
\right]\mathbf 1\{\theta\in[-R,R]^d\}.
$$

The data are generated so that several coordinates are naturally pulled toward the box limits.  This is not meant to be adversarial; it is the regime in which an exact constrained Gaussian RGO is useful.  Projected PGLA can create boundary atoms, and Prox-MALA can waste Gaussian proposals outside the feasible set.  The composite sampler's box RGO samples from the truncated Gaussian directly, so it neither leaves the set nor projects onto a boundary atom.

In [ ]:
def make_box_logistic_problem(seed=20, paper_run=False):
    rng = np.random.default_rng(seed)
    if paper_run:
        d, n, n_ref, n_main, n_pilot = 24, 360, 16_000, 6_000, 1_200
    else:
        d, n, n_ref, n_main, n_pilot = 12, 150, 4_000, 1_700, 350

    rho = 0.55
    idx = np.arange(d)
    cov = rho ** np.abs(idx.reshape(-1, 1) - idx.reshape(1, -1))
    A = rng.multivariate_normal(np.zeros(d), cov, size=n)
    A = A / np.linalg.norm(A, axis=0) * math.sqrt(n)

    R = 0.35
    lower = -R * np.ones(d)
    upper = R * np.ones(d)

    theta_true = np.zeros(d)
    theta_true[:4] = 3.0 * np.array([1.0, -1.0, 1.0, -1.0])
    y = rng.binomial(1, expit(A @ theta_true))

    tau = 0.20

    f = LogisticPotential(A, y, tau)
    box = BoxRGO(lower, upper)
    x_star = solve_prox_map(f, box, f.beta, x0=np.clip(theta_true, lower, upper), n_iter=4_000)
    return f, box, x_star, theta_true, n_ref, n_main, n_pilot


f_box, box, x_star_box, theta_true_box, n_ref_box, n_main_box, n_pilot_box = make_box_logistic_problem(
    seed=42, paper_run=PAPER_RUN
)
print(f"d={f_box.dim}, alpha={f_box.alpha:.3g}, beta={f_box.beta:.3g}, kappa={f_box.beta/f_box.alpha:.1f}")
active = np.sum(np.isclose(x_star_box, box.lower) | np.isclose(x_star_box, box.upper))
print(f"MAP touches boundary in {active} coordinates")

ref_box = build_reference_prox_mala(
    f_box, box, x_star_box, n_steps=n_ref_box, step_multiplier=0.05, seeds=[201, 202, 203]
)
ref_box_samples = ref_box["samples"]
ref_box_mean = ref_box_samples.mean(axis=0)
print(
    f"Reference: {len(ref_box_samples)} retained samples, "
    f"acceptance={acceptance_mean(ref_box):.3f}, "
    f"feasible proposal rate={feasible_mean(ref_box):.3f}, elapsed={ref_box['elapsed']:.2f}s"
)

In [ ]:
inner_steps_box = 8
composite_grid_box = [0.25, 0.5, 1.0, 2.0, 4.0, 8.0]
langevin_grid_box = [0.025, 0.05, 0.10, 0.20, 0.40, 0.80, 1.20]

tune_comp_box, mult_comp_box = run_tuning_grid(
    "Composite sampler",
    composite_grid_box,
    lambda mult, i: composite_sampler(
        f_box,
        box,
        h=mult / f_box.beta,
        n_steps=n_pilot_box,
        inner_steps=inner_steps_box,
        x_star=x_star_box,
        beta=f_box.beta,
        rng=np.random.default_rng(600 + i),
    ),
    ref_box_mean,
    min_acceptance=0.15,
)

tune_pmala_box, mult_pmala_box = run_tuning_grid(
    "Prox-MALA",
    langevin_grid_box,
    lambda mult, i: prox_mala(
        f_box,
        box,
        step=mult / f_box.beta,
        n_steps=n_pilot_box,
        rng=np.random.default_rng(700 + i),
        x0=x_star_box,
    ),
    ref_box_mean,
    min_acceptance=0.10,
)

tune_pgla_box, mult_pgla_box = run_tuning_grid(
    "PGLA (no MH)",
    langevin_grid_box,
    lambda mult, i: proximal_gradient_langevin(
        f_box,
        box,
        step=mult / f_box.beta,
        n_steps=n_pilot_box,
        rng=np.random.default_rng(800 + i),
        x0=x_star_box,
    ),
    ref_box_mean,
)

tuning_box = pd.concat([tune_comp_box, tune_pmala_box, tune_pgla_box], ignore_index=True)
plot_tuning(tuning_box, "Step-size tuning: active box logistic target", "box_logistic_step_tuning")
tuning_box

In [ ]:
comp_box = composite_sampler(
    f_box,
    box,
    h=mult_comp_box / f_box.beta,
    n_steps=n_main_box,
    inner_steps=inner_steps_box,
    x_star=x_star_box,
    beta=f_box.beta,
    rng=np.random.default_rng(900),
)

pmala_box = prox_mala(
    f_box,
    box,
    step=mult_pmala_box / f_box.beta,
    n_steps=n_main_box,
    rng=np.random.default_rng(901),
    x0=x_star_box,
)

pgla_box = proximal_gradient_langevin(
    f_box,
    box,
    step=mult_pgla_box / f_box.beta,
    n_steps=n_main_box,
    rng=np.random.default_rng(902),
    x0=x_star_box,
)

summary_box = pd.DataFrame(
    [
        summarize_chain("Composite sampler", comp_box, ref_box_samples, ref_box_mean, artifact="box_boundary", box=box),
        summarize_chain("Prox-MALA", pmala_box, ref_box_samples, ref_box_mean, artifact="box_boundary", box=box),
        summarize_chain("PGLA (no MH)", pgla_box, ref_box_samples, ref_box_mean, artifact="box_boundary", box=box),
    ]
)
summary_box.to_json("Box-constrained_logistic__summary.json", orient="records", indent=2)
summary_box

In [ ]:
plot_running_rmse(
    [
        ("Composite sampler", comp_box),
        ("Prox-MALA", pmala_box),
        ("PGLA (no MH)", pgla_box),
    ],
    ref_box_mean,
    title="Box-constrained logistic regression: posterior mean error",
    filename="box_logistic_mean_rmse",
)

plot_metric_bars(
    summary_box,
    "boundary-hit fraction",
    title="Active box logistic: boundary atoms",
    filename="box_logistic_boundary_artifact",
    ylabel="fraction of coordinates exactly on boundary",
    ylim=(0.0, max(0.04, 1.15 * float(summary_box["boundary-hit fraction"].max()))),
)

feasible_df = summary_box[summary_box["method"].isin(["Prox-MALA"])].copy()
if len(feasible_df) and np.isfinite(feasible_df["feasible proposal rate"].iloc[0]):
    plot_metric_bars(
        feasible_df,
        "feasible proposal rate",
        title="Active box logistic: Prox-MALA feasible proposal rate",
        filename="box_logistic_pmala_feasible_rate",
        ylabel="fraction of Gaussian proposals inside the box",
        ylim=(0.0, 1.05),
    )

fig, axes = plt.subplots(1, 4, figsize=(12.8, 3.6), sharex=True, sharey=True)
plot_sets = [
    ("Reference", ref_box_samples),
    ("Composite sampler", comp_box["samples"]),
    ("Prox-MALA", pmala_box["samples"]),
    ("PGLA (no MH)", pgla_box["samples"]),
]
for idx, (ax, (name, samples)) in enumerate(zip(axes, plot_sets)):
    st = style_for(name, idx)
    samples = np.asarray(samples)
    keep = slice(None, None, max(1, len(samples) // 700))
    ax.scatter(samples[keep, 0], samples[keep, 1], s=10, alpha=0.42, color=st["color"], edgecolors="none")
    ax.axvline(box.lower[0], color="0.45", lw=1.0, ls=":")
    ax.axvline(box.upper[0], color="0.45", lw=1.0, ls=":")
    ax.axhline(box.lower[1], color="0.45", lw=1.0, ls=":")
    ax.axhline(box.upper[1], color="0.45", lw=1.0, ls=":")
    ax.set_title(name)
    ax.set_xlabel(r"$\theta_0$")
    if ax is axes[0]:
        ax.set_ylabel(r"$\theta_1$")
    ax.grid(True, alpha=0.18)
fig.suptitle("Active box logistic: first two coordinates", y=1.03, fontsize=12, fontweight="semibold")
save_figure(fig, "box_logistic_scatter")


bias_rows = []
n_bias_box = n_pilot_box if not PAPER_RUN else 2 * n_pilot_box
for i, mult in enumerate(langevin_grid_box):
    res = proximal_gradient_langevin(
        f_box,
        box,
        step=mult / f_box.beta,
        n_steps=n_bias_box,
        rng=np.random.default_rng(1600 + i),
        x0=x_star_box,
    )
    bias_rows.append({
        "multiplier": float(mult),
        "mean RMSE": mean_rmse(res["samples"], ref_box_mean),
        "boundary-hit fraction": boundary_hit_fraction(res["samples"], box),
        "selected": float(mult) == float(mult_pgla_box),
    })
pgla_bias_box = pd.DataFrame(bias_rows)
pgla_bias_box.to_json("Box-constrained_logistic__pgla_bias_curve.json", orient="records", indent=2)
plot_bias_tradeoff(
    pgla_bias_box,
    "boundary-hit fraction",
    title="Active box logistic: PGLA boundary bias grows with step size",
    filename="box_logistic_pgla_bias_tradeoff",
    artifact_label="boundary-hit fraction",
)
pgla_bias_box

# Experiment 3: Gaussian restricted to a box

Here we do a scaling experiment on a simple problem: sampling from Gaussian restricted to the box $[-R,R]^d$. For generality, we let $\kappa = 1$ for our problem. We study how the iteration complexity scales with the dimension $d$ provided that any of the three algorithms achieves a sliced $W_2$ distance to $\pi$ bounded by $\varepsilon = 0.05$.

$d$ here comes from the sequence {4, 8, 16, 32, 64, 128, 256, 512}. For each $d$, we run experiments for 3 seeds.

In [ ]:
class GaussianPotential:
    def __init__(self, d, tau):
        self._d = int(d); self._tau = float(tau)
    @property
    def dim(self): return self._d
    @property
    def alpha(self): return self._tau
    @property
    def beta(self): return self._tau
    def value(self, x):
        x = np.asarray(x, dtype=float)
        return 0.5 * self._tau * float(np.dot(x, x))
    def grad(self, x):
        return self._tau * np.asarray(x, dtype=float)


def make_gaussian_box_problem(d, tau=1.0, R=1.0):
    """π ∝ N(0, τ⁻¹I) restricted to [-R,R]^d. κ = 1 by construction; x* = 0."""
    return GaussianPotential(d, tau), BoxRGO(-R*np.ones(d), R*np.ones(d)), np.zeros(d)


def sample_gaussian_box_exact(d, tau, R, n_samples, rng):
    """Exact i.i.d. draws from the truncated-normal product target -- the
    reference for the scaling experiment, no MCMC needed."""
    sigma = 1.0 / math.sqrt(tau)
    return truncnorm_stable_rvs(0.0, sigma, -R, R, rng, size=(n_samples, d))

In [ ]:
def run_dimension_scaling(
    d_grid, tau=1.0, R=1.0,
    n_outer_comp=5000, n_iter_pmala=15000, n_iter_pgla=25000,
    inner_steps=8, n_ref=4000, n_proj=64, seeds=(0, 1, 2),
    threshold=0.05, verbose=True,
):
    """Run all three samplers at each d, record running sliced W2 curves and
    the gradient-evaluation cost to first cross `threshold`.

    Step sizes follow each method's optimal scaling rate, with hand-picked
    constants:
        Composite : h    = 1.0 / (beta * sqrt(d))    (Algorithm 1's prescription)
        Prox-MALA : step = 0.2 / (beta * d)          (smaller -- many proposals
                                                       leave the box at high d)
        PGLA      : step = 0.5 / (beta * d)          (Langevin)
    """
    curves, rows = [], []
    for d in d_grid:
        f, box, x_star = make_gaussian_box_problem(d, tau=tau, R=R)
        beta = f.beta
        ref_samples = sample_gaussian_box_exact(
            d, tau, R, n_ref, np.random.default_rng(7777 + d)
        )
        h_comp     = 1.0 / (beta * math.sqrt(d))
        step_pmala = 0.2 / (beta * d)
        step_pgla  = 0.5 / (beta * d)

        for seed in seeds:
            t0 = time.perf_counter()
            res_c = composite_sampler(
                f, box, h=h_comp, n_steps=n_outer_comp, inner_steps=inner_steps,
                x_star=x_star, beta=beta,
                rng=np.random.default_rng(seed * 1000 + d),
            )
            res_p = prox_mala(
                f, box, step=step_pmala, n_steps=n_iter_pmala,
                rng=np.random.default_rng(seed * 1000 + d + 100), x0=x_star,
            )
            res_g = proximal_gradient_langevin(
                f, box, step=step_pgla, n_steps=n_iter_pgla,
                rng=np.random.default_rng(seed * 1000 + d + 200), x0=x_star,
            )
            for name, res in [("Composite sampler", res_c),
                               ("Prox-MALA", res_p),
                               ("PGLA", res_g)]:
                grid, w2 = running_sliced_w2(
                    res["samples"], ref_samples, n_proj=n_proj,
                    rng=np.random.default_rng(99 + d + seed),
                )
                ge = grid * res.get("work_per_sample", 1)
                for k in range(len(grid)):
                    curves.append({"d": d, "method": name, "seed": seed,
                                   "grad_evals": int(ge[k]),
                                   "sliced_w2": float(w2[k])})
                hits = np.where(w2 <= threshold)[0]
                cost = int(ge[hits[0]]) if len(hits) else np.nan
                rows.append({"d": d, "method": name, "seed": seed,
                             "cost_to_threshold": cost,
                             "final_w2": float(w2[-1])})
            if verbose:
                print(f"  d={d:4d} seed={seed}: {time.perf_counter()-t0:.1f}s")
    return pd.DataFrame(curves), pd.DataFrame(rows)

In [ ]:
THRESHOLD = 0.05
D_GRID = [4, 8, 16, 32, 64, 128, 256, 512]
SEEDS = (2026, 506, 111) # can be any random seed

print("Running dimension-scaling experiment (Gaussian on box, kappa = 1)...")
curves_df, rows_df = run_dimension_scaling(
    d_grid=D_GRID, tau=1.0, R=1.0,
    n_outer_comp=5000, n_iter_pmala=100000, n_iter_pgla=100000,
    inner_steps=8, n_ref=4000, n_proj=64,
    seeds=SEEDS, threshold=THRESHOLD,
)
curves_df.to_json("scaling_curves.json", orient="records", indent=2)
rows_df.to_json("scaling_costs.json", orient="records", indent=2)

print("\n--- Cost to threshold (median over seeds, gradient evaluations) ---")
print((rows_df.groupby(["d", "method"])["cost_to_threshold"]
       .median().unstack("method")))
print("\n--- Final sliced W2 (median over seeds) ---")
print((rows_df.groupby(["d", "method"])["final_w2"]
       .median().unstack("method")).round(3))

In [ ]:
%%capture
# Figure 1 at every d in the grid: produces scaling_running_w2_d{d}.pdf (and .png)
for d in D_GRID:
    plot_running_w2_at_d(
        curves_df, target_d=d, threshold=THRESHOLD,
        title=rf"Sliced W2 convergence on Gaussian-on-box, $d={d}$, $\kappa=1$",
        filename=f"scaling_running_w2_d{d}",
    )

In [ ]:
slope_df = plot_cost_vs_d_slope(
    rows_df, threshold=THRESHOLD,
    title=r"Cost-to-threshold vs dimension (Gaussian on box, $\kappa=1$)",
    filename="scaling_cost_vs_d", fit_min_d=32,
)